# Sesión 4C · Selenium paso a paso

El notebook anterior extrae 60 ofertas en una celda. Va rápido **a propósito**,
porque muestra el resultado final.

Este va al revés: vamos a sacar **un solo dato a la vez**, mirando qué
devuelve cada instrucción, hasta entender exactamente qué está pasando. Recién
al final escribimos la función.

El orden es el que se usa de verdad cuando uno se enfrenta a una página nueva:

1. Abrir el navegador y mirar
2. Encontrar **una** pieza
3. Abrirla y ver qué tiene adentro
4. Sacar **un** campo
5. Sacar los demás
6. Recién ahí: repetir para todas

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re

## Paso 1 · Abrir el navegador

Lo dejamos **visible** (`--headless` comentado) para ir viendo lo que pasa.

In [2]:
opciones = Options()
# opciones.add_argument("--headless=new")   # descomenta para ocultarlo
opciones.add_argument("--window-size=1400,900")
opciones.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")

Navegador abierto


In [3]:
navegador.get("https://www.bumeran.com.pe/empleos-busqueda-analista-de-datos.html")

WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/empleos/']"))
)
time.sleep(3)

print("Título:", navegador.title)

Título: analista de datos: Empleos - Página 1 | Bumeran Perú


## Paso 2 · `find_element` y `find_elements`

Solo hay dos instrucciones para buscar, y se diferencian en una letra:

| Instrucción | Devuelve | Si no encuentra |
|---|---|---|
| `find_element` (singular) | **un** elemento | lanza error |
| `find_elements` (plural) | **una lista** | lista vacía `[]` |

Empecemos por el singular.

In [4]:
primer_h2 = navegador.find_element(By.TAG_NAME, "h2")

print("Tipo de objeto:", type(primer_h2))
print("Texto          :", primer_h2.text)

Tipo de objeto: <class 'selenium.webdriver.remote.webelement.WebElement'>
Texto          : Bolsa de empleo


Eso que devolvió es un **WebElement**: no es texto, es una *referencia* a un
pedazo de la página abierta en el navegador. Para sacarle el contenido hay que
pedírselo.

### Ahora el plural

In [5]:
todos_los_h2 = navegador.find_elements(By.TAG_NAME, "h2")

print("Tipo   :", type(todos_los_h2))
print("Cuántos:", len(todos_los_h2))

Tipo   : <class 'list'>
Cuántos: 21


### La diferencia cuando no existe

Esto es importante: el plural **no falla**, devuelve una lista vacía. Por eso
se usa cuando no estás seguro de que el elemento esté.

In [6]:
inexistente = navegador.find_elements(By.TAG_NAME, "marquee")
print("find_elements con algo que no existe:", inexistente)

find_elements con algo que no existe: []


In [7]:
try:
    navegador.find_element(By.TAG_NAME, "marquee")
except Exception as e:
    print("find_element con algo que no existe:", type(e).__name__)

find_element con algo que no existe: NoSuchElementException


## Paso 3 · Las formas de buscar (`By`)

El primer argumento dice **cómo** buscar. Todas estas hacen lo mismo aquí:

In [8]:
print("Por etiqueta       :", len(navegador.find_elements(By.TAG_NAME, "h2")))
print("Por selector CSS   :", len(navegador.find_elements(By.CSS_SELECTOR, "h2")))
print("Por XPath          :", len(navegador.find_elements(By.XPATH, "//h2")))

Por etiqueta       : 21
Por selector CSS   : 21
Por XPath          : 21


| Forma | Cuándo usarla |
|---|---|
| `By.ID` | hay un `id`. Es lo mejor, pero casi nunca existe |
| `By.TAG_NAME` | quieres todos los `h2`, `a`, `table`… |
| `By.CSS_SELECTOR` | lo más usado. Permite combinar: `a[href]`, `div.caja p` |
| `By.XPATH` | lo más potente; sirve para subir al padre o buscar por texto |
| `By.CLASS_NAME` | útil **solo** si la clase tiene significado |

En este sitio las clases son basura generada (`sc-ekQYnd`), así que quedan
descartadas.

## Paso 4 · Encontrar las tarjetas de oferta

Cada oferta es un enlace `<a>`. Pero la página tiene muchos enlaces:

In [9]:
todos_los_enlaces = navegador.find_elements(By.CSS_SELECTOR, "a[href]")
print("Enlaces en la página:", len(todos_los_enlaces))

Enlaces en la página: 47


### Mirar unos cuantos para encontrar el patrón

In [10]:
for enlace in todos_los_enlaces[:8]:
    print(" ", (enlace.get_attribute("href") or "")[:78])

  https://www.bumeran.com.pe/
  https://www.bumeran.com.pe/signin
  https://www.bumeran.com.pe/login
  https://www.bumeran.com.pe/listado-empresas
  https://www.bumeran.com.pe/salarios
  https://www.bumeran.com.pe/blog/
  https://www.bumeran.com.pe/empleos-busqueda-analista-de-datos.html?relevantes=
  https://www.bumeran.com.pe/empleos-busqueda-analista-de-datos.html?recientes=t


### El patrón

Las ofertas siguen siempre la misma forma:

```
/empleos/  algo-descriptivo  -  1118448397  .html
           ↑                    ↑
           título del puesto    número de aviso
```

Se traduce a esta expresión regular:

```python
r"/empleos/.+-\d+\.html"
```

| Parte | Significa |
|---|---|
| `/empleos/` | tal cual, esas letras |
| `.+` | cualquier cosa, al menos un carácter |
| `-` | un guion |
| `\d+` | uno o más dígitos |
| `\.html` | el punto y "html" |

In [11]:
tarjetas = []
for enlace in todos_los_enlaces:
    direccion = enlace.get_attribute("href") or ""
    if re.search(r"/empleos/.+-\d+\.html", direccion):
        tarjetas.append(enlace)

print("Tarjetas de oferta encontradas:", len(tarjetas))

Tarjetas de oferta encontradas: 20


## Paso 5 · Radiografía de UNA tarjeta

Nos quedamos con la primera y la estudiamos entera. Todo lo que sigue es
sobre ella.

In [12]:
tarjeta = tarjetas[0]

print("Es un:", tarjeta.tag_name)
print("Lleva a:", tarjeta.get_attribute("href"))

Es un: a
Lleva a: https://www.bumeran.com.pe/empleos/analista-de-datos-fractal-soluciones-it-1118447809.html


### Su texto completo

In [13]:
print(tarjeta.text)

Actualizado hace 9 horas
Analista de Datos
FRACTAL SOLUCIONES IT
Alta revisión de perfiles
FRACTAL Soluciones TI, empresa peruana de Consultoría en Gestión de Tecnología de la Información con más de 12 años en el mercado. Nos encontramos en la búsqueda de un Analista de Datos TI – AWS & Databricks para un proyecto de 6 meses con uno de nuestros clientes del sector bancario con el siguiente perfil: Requisitos Indispensables: Bachiller Técnico o Superior en Ingeniería de Sistemas, Informática, Ciencia de Datos o afines. Experiencia en Proyectos comerciales, generación de insights, productos digitales Experiencia en Databricks. Dominio de Bases de Datos relacionales y no relacionales. Experiencia intermedia en servicios Cloud con AWS. Manejo intermedio de Power BI para generación de reportes y dashboards.Beneficios: Capacitaciones en plataforma AWS-AMAZON. Descuentos corporativos. Actividades de bienestar y más.Conócenos en: https://www.fractal.com.pe/
Múltiples vacantes
Lima, Lima
Híbrid

### ¿Qué etiquetas tiene adentro?

Una tarjeta es una caja con más cajas. Veamos de qué está hecha:

In [14]:
for etiqueta in ["h1", "h2", "h3", "h4", "p", "span", "img"]:
    encontrados = tarjeta.find_elements(By.TAG_NAME, etiqueta)
    if encontrados:
        print(f"  <{etiqueta}>: {len(encontrados)}")

  <h2>: 1
  <h3>: 4
  <p>: 2
  <span>: 7


**Importante:** `tarjeta.find_elements(...)` busca **solo dentro de esa
tarjeta**, no en toda la página. Buscar desde un elemento en vez de desde el
navegador es lo que permite procesar una oferta a la vez sin mezclarlas.

## Paso 6 · Sacar el primer campo: el puesto

Hay un solo `<h2>` por tarjeta, y es el título del puesto.

In [15]:
elemento_puesto = tarjeta.find_element(By.TAG_NAME, "h2")
elemento_puesto

<selenium.webdriver.remote.webelement.WebElement (session="b2ce7fb94945d3ed0a0faae131cab045", element="f.6922745AB35470B0CF4F74377B67DE78.d.F460E2E73342EEA66E14342F781E04F0.e.119")>

Eso imprime el objeto, no el texto. Para el texto:

In [16]:
puesto = elemento_puesto.text
print(repr(puesto))

'Analista de Datos'


Ya tenemos un dato. En una línea:

In [17]:
puesto = tarjeta.find_element(By.TAG_NAME, "h2").text
puesto

'Analista de Datos'

## Paso 7 · Los `<h3>`: aquí está la trampa

El resto de los datos vive en etiquetas `<h3>`. Veámoslos con su posición:

In [18]:
h3 = tarjeta.find_elements(By.TAG_NAME, "h3")

for i, elemento in enumerate(h3):
    print(f"  h3[{i}] = {elemento.text!r}")

  h3[0] = 'Actualizado hace 9 horas'
  h3[1] = 'FRACTAL SOLUCIONES IT'
  h3[2] = 'Lima, Lima'
  h3[3] = 'Híbrido'


Se ve clarísimo:

- `h3[0]` → fecha
- `h3[1]` → empresa
- `h3[2]` → ubicación
- `h3[3]` → modalidad

La tentación es escribir `h3[2].text` para la ubicación y listo.

**No lo hagas.** Vamos a comprobar por qué.

### Comprobemos con otras tarjetas antes de confiar

Esta es la costumbre que separa un scraper que aguanta de uno que se cae:
**nunca generalices desde un solo ejemplo.**

In [19]:
for numero in [0, 1, 2, 3, 4, 5]:
    textos = [e.text for e in tarjetas[numero].find_elements(By.TAG_NAME, "h3")]
    print(f"  Tarjeta {numero} ({len(textos)} h3): {textos}")

  Tarjeta 0 (4 h3): ['Actualizado hace 9 horas', 'FRACTAL SOLUCIONES IT', 'Lima, Lima', 'Híbrido']


  Tarjeta 1 (4 h3): ['Publicado hace 9 días', 'Confidencial', 'Cercado De Lima, Lima', 'Presencial']


  Tarjeta 2 (4 h3): ['Publicado hace 7 horas', 'OSLO', 'San Borja, Lima', 'Híbrido']


  Tarjeta 3 (4 h3): ['Actualizado hace 2 días', 'INFORMATICA DELTA S.A.C.', 'Lima, Lima', 'Híbrido']


  Tarjeta 4 (5 h3): ['Actualizado hace 2 días', 'Hitss Perú', 'La Victoria, Lima', 'Presencial', 'Apto discapacidad']


  Tarjeta 5 (4 h3): ['Actualizado hace 2 días', 'Inetum Perú', 'San Isidro, Lima', 'Presencial']


### Ahí está el problema

Algunas tarjetas tienen un `<h3>` **extra** con la calificación de la empresa
(`3.1`, `4.4`). En esas, todo se corre un lugar:

| | Tarjeta sin calificación | Tarjeta con calificación |
|---|---|---|
| `h3[0]` | fecha | fecha |
| `h3[1]` | empresa | empresa |
| `h3[2]` | **ubicación** | **3.1** ← calificación |
| `h3[3]` | modalidad | ubicación |

Si usas `h3[2]` para la ubicación, en esas tarjetas guardas un número.

Y no da error: **guarda el dato equivocado en silencio**. Ese es el peor tipo
de bug, porque llega hasta tu informe sin que nadie lo note.

### Contemos cuántas están afectadas

In [20]:
conteo = {}
for t in tarjetas:
    cuantos = len(t.find_elements(By.TAG_NAME, "h3"))
    conteo[cuantos] = conteo.get(cuantos, 0) + 1

print("Cantidad de <h3> por tarjeta:", conteo)

Cantidad de <h3> por tarjeta: {4: 13, 5: 5, 6: 2}


## Paso 8 · La solución: buscar por contenido, no por posición

Los dos primeros sí son estables, esos se pueden tomar por índice:

In [21]:
fecha = h3[0].text
empresa = h3[1].text

print("Fecha  :", fecha)
print("Empresa:", empresa)

Fecha  : Actualizado hace 9 horas
Empresa: FRACTAL SOLUCIONES IT


### La modalidad: es uno de tres valores conocidos

En vez de preguntar "¿cuál es el cuarto?", preguntamos "¿cuál de estos textos
es una modalidad?".

In [22]:
MODALIDADES = {"Presencial", "Remoto", "Híbrido"}

textos_h3 = [e.text for e in h3]

modalidad = None
for texto in textos_h3:
    if texto in MODALIDADES:
        modalidad = texto

print("Modalidad:", modalidad)

Modalidad: Híbrido


### La ubicación: se reconoce por la coma

Siempre viene como `"Distrito, Ciudad"`. Ni la fecha, ni la calificación, ni
la modalidad llevan coma.

In [23]:
ubicacion = None
for texto in textos_h3:
    if "," in texto:
        ubicacion = texto

print("Ubicación:", ubicacion)

Ubicación: Lima, Lima


### Probemos las dos reglas en las tarjetas problemáticas

Volvamos a las mismas seis de antes:

In [24]:
for numero in [0, 1, 2, 3, 4, 5]:
    textos = [e.text for e in tarjetas[numero].find_elements(By.TAG_NAME, "h3")]
    ubi = next((t for t in textos if "," in t), None)
    mod = next((t for t in textos if t in MODALIDADES), None)
    print(f"  Tarjeta {numero}: ubicación={ubi!r:28} modalidad={mod!r}")

  Tarjeta 0: ubicación='Lima, Lima'                 modalidad='Híbrido'


  Tarjeta 1: ubicación='Cercado De Lima, Lima'      modalidad='Presencial'


  Tarjeta 2: ubicación='San Borja, Lima'            modalidad='Híbrido'


  Tarjeta 3: ubicación='Lima, Lima'                 modalidad='Híbrido'


  Tarjeta 4: ubicación='La Victoria, Lima'          modalidad='Presencial'
  Tarjeta 5: ubicación='San Isidro, Lima'           modalidad='Presencial'


Ahora sí: funciona igual tenga la tarjeta 4, 5 o 6 elementos.

**La regla general:** si la posición de un dato puede cambiar, búscalo por lo
que **es**, no por dónde está.

## Paso 9 · El enlace: `get_attribute`

`.text` da lo que se ve. Para lo que **no** se ve —la dirección de un enlace,
la fuente de una imagen— se usa `get_attribute`.

In [25]:
print("texto        :", tarjeta.text[:40].replace("\n", " / "))
print("href         :", tarjeta.get_attribute("href"))
print("target       :", tarjeta.get_attribute("target"))
print("class        :", tarjeta.get_attribute("class"))

texto        : Actualizado hace 9 horas / Analista de Dat
href         : https://www.bumeran.com.pe/empleos/analista-de-datos-fractal-soluciones-it-1118447809.html
target       : _blank
class        : sc-ekQYnd bAylLX


### Sacar el número de aviso de la URL

Ese número identifica la oferta. Sirve para no guardar duplicados.

In [26]:
url = tarjeta.get_attribute("href")
numero_aviso = re.search(r"-(\d+)\.html", url).group(1)

print("URL   :", url)
print("Aviso :", numero_aviso)

URL   : https://www.bumeran.com.pe/empleos/analista-de-datos-fractal-soluciones-it-1118447809.html
Aviso : 1118447809


## Paso 10 · La descripción

Está en un `<p>`. Pero hay más de uno:

In [27]:
parrafos = tarjeta.find_elements(By.TAG_NAME, "p")

for i, p in enumerate(parrafos):
    print(f"  p[{i}] ({len(p.text)} caracteres): {p.text[:60]!r}")

  p[0] (25 caracteres): 'Alta revisión de perfiles'
  p[1] (872 caracteres): 'FRACTAL Soluciones TI, empresa peruana de Consultoría en Ges'


El primero a veces es una etiqueta corta ("Alta revisión de perfiles"). La
descripción es **el más largo**:

In [28]:
descripcion = max((p.text for p in parrafos), key=len, default="")
print(descripcion[:250])

FRACTAL Soluciones TI, empresa peruana de Consultoría en Gestión de Tecnología de la Información con más de 12 años en el mercado. Nos encontramos en la búsqueda de un Analista de Datos TI – AWS & Databricks para un proyecto de 6 meses con uno de nue


## Paso 11 · Juntar todo en un diccionario

Ya tenemos los seis campos. Los guardamos juntos:

In [29]:
oferta = {
    "puesto": puesto,
    "empresa": empresa,
    "fecha": fecha,
    "ubicacion": ubicacion,
    "modalidad": modalidad,
    "aviso": numero_aviso,
    "url": url,
}

oferta

{'puesto': 'Analista de Datos',
 'empresa': 'FRACTAL SOLUCIONES IT',
 'fecha': 'Actualizado hace 9 horas',
 'ubicacion': 'Lima, Lima',
 'modalidad': 'Híbrido',
 'aviso': '1118447809',
 'url': 'https://www.bumeran.com.pe/empleos/analista-de-datos-fractal-soluciones-it-1118447809.html'}

**Eso es una fila de nuestra tabla final.** Todo lo que sigue es repetir esto.

## Paso 12 · Hacerlo con la segunda tarjeta

Copiamos y pegamos, cambiando la tarjeta:

In [30]:
otra = tarjetas[1]

h3_otra = otra.find_elements(By.TAG_NAME, "h3")
textos_otra = [e.text for e in h3_otra]

oferta_2 = {
    "puesto": otra.find_element(By.TAG_NAME, "h2").text,
    "empresa": h3_otra[1].text,
    "fecha": h3_otra[0].text,
    "ubicacion": next((t for t in textos_otra if "," in t), None),
    "modalidad": next((t for t in textos_otra if t in MODALIDADES), None),
    "aviso": re.search(r"-(\d+)\.html", otra.get_attribute("href")).group(1),
    "url": otra.get_attribute("href"),
}

oferta_2

{'puesto': 'Analista de Datos',
 'empresa': 'Confidencial',
 'fecha': 'Publicado hace 9 días',
 'ubicacion': 'Cercado De Lima, Lima',
 'modalidad': 'Presencial',
 'aviso': '1118440330',
 'url': 'https://www.bumeran.com.pe/empleos/analista-de-datos-1118440330.html'}

### Copiar y pegar es la señal

Acabamos de escribir lo mismo dos veces cambiando una palabra. Esa es
exactamente la señal de que toca una función.

No escribimos la función al principio porque no sabíamos qué iba adentro.
Primero se resuelve a mano, después se empaqueta.

## Paso 13 · La función

Es el mismo código de arriba, con la tarjeta como parámetro.

In [31]:
def leer_tarjeta(tarjeta):
    """Convierte una tarjeta de oferta de Bumeran en un diccionario."""
    h3 = tarjeta.find_elements(By.TAG_NAME, "h3")
    textos = [e.text for e in h3]
    url = tarjeta.get_attribute("href")

    # Los dos primeros h3 son estables; el resto se busca por contenido
    return {
        "puesto":    tarjeta.find_element(By.TAG_NAME, "h2").text,
        "fecha":     textos[0] if len(textos) > 0 else None,
        "empresa":   textos[1] if len(textos) > 1 else None,
        "ubicacion": next((t for t in textos if "," in t), None),
        "modalidad": next((t for t in textos if t in MODALIDADES), None),
        "aviso":     re.search(r"-(\d+)\.html", url).group(1),
        "url":       url,
    }


leer_tarjeta(tarjetas[0])

{'puesto': 'Analista de Datos',
 'fecha': 'Actualizado hace 9 horas',
 'empresa': 'FRACTAL SOLUCIONES IT',
 'ubicacion': 'Lima, Lima',
 'modalidad': 'Híbrido',
 'aviso': '1118447809',
 'url': 'https://www.bumeran.com.pe/empleos/analista-de-datos-fractal-soluciones-it-1118447809.html'}

### Comprobar que da lo mismo que a mano

In [32]:
print("¿Coincide con lo que sacamos paso a paso?")
print("  ", leer_tarjeta(tarjetas[0]) == oferta)

¿Coincide con lo que sacamos paso a paso?
   True


## Paso 14 · Recién ahora, el bucle

In [33]:
resultados = []

for t in tarjetas:
    resultados.append(leer_tarjeta(t))

print("Ofertas leídas:", len(resultados))

Ofertas leídas: 20


In [34]:
empleos = pd.DataFrame(resultados)
empleos

,puesto,fecha,empresa,ubicacion,modalidad,aviso,url
0,Analista de Datos,Actualizado hace 9 horas,FRACTAL SOLUCIONES IT,"Lima, Lima",Híbrido,1118447809,https://www.bumeran.com.pe/empleos/analista-de...
1,Analista de Datos,Publicado hace 9 días,Confidencial,"Cercado De Lima, Lima",Presencial,1118440330,https://www.bumeran.com.pe/empleos/analista-de...
2,Analista de Datos Operativos,Publicado hace 7 horas,OSLO,"San Borja, Lima",Híbrido,1118453111,https://www.bumeran.com.pe/empleos/analista-de...
3,430 Analista de Datos,Actualizado hace 2 días,INFORMATICA DELTA S.A.C.,"Lima, Lima",Híbrido,1118448331,https://www.bumeran.com.pe/empleos/430-analist...
4,Analista de Explotación de Datos,Actualizado hace 2 días,Hitss Perú,"La Victoria, Lima",Presencial,1118447643,https://www.bumeran.com.pe/empleos/analista-de...
5,Analista de datos junior,Actualizado hace 2 días,Inetum Perú,"San Isidro, Lima",Presencial,1118442441,https://www.bumeran.com.pe/empleos/analista-de...
6,Analista de Datos,Actualizado hace 6 días,IBT GROUP,"San Isidro, Lima",Presencial,1118432292,https://www.bumeran.com.pe/empleos/analista-de...
7,Analista de Gobierno y Calidad de Datos,Publicado hace 3 días,Rimac Cia de Seguros y Reaseguros S.A.,"San Isidro, Lima",Híbrido,1118448706,https://www.bumeran.com.pe/empleos/analista-de...
8,Analista de Gestión de Datos Maestros,Publicado hace 5 días,CL SELECTION/DIVISION IT,"Lima, Lima",Presencial,1118447276,https://www.bumeran.com.pe/empleos/analista-de...
9,Analista de Datos Junior. (Hibrido),Actualizado hace 2 días,CSTI Corp,"Miraflores, Lima",Híbrido,1118446068,https://www.bumeran.com.pe/empleos/analista-de...


### Revisar que no haya campos vacíos

Último control antes de darlo por bueno:

In [35]:
empleos.isna().sum()

puesto       0
fecha        0
empresa      0
ubicacion    0
modalidad    0
aviso        0
url          0
dtype: int64

In [36]:
empleos["modalidad"].value_counts(dropna=False)

modalidad
Híbrido       10
Presencial    10
Name: count, dtype: int64

## Paso 15 · Cerrar el navegador

Siempre. Si no, queda un Chrome invisible consumiendo memoria.

In [37]:
navegador.quit()
print("Cerrado")

Cerrado


---
## Los errores que vas a ver

### `NoSuchElementException`

Usaste `find_element` (singular) y no existe. Solución: usa el plural y
revisa si la lista está vacía.

```python
encontrados = tarjeta.find_elements(By.TAG_NAME, "h4")
valor = encontrados[0].text if encontrados else None
```

### `StaleElementReferenceException`

Guardaste un elemento, la página se volvió a dibujar, y tu referencia apunta
a algo que ya no existe. Pasa al cambiar de página y seguir usando la lista
vieja.

**Solución:** vuelve a buscar los elementos después de cada navegación. Nunca
guardes WebElements entre páginas — guarda el **texto** que ya extrajiste.

### `TimeoutException`

El `WebDriverWait` esperó y nunca apareció. O cambió el selector, o la página
no cargó. Ponlo en visible y mira qué pasa.

### La lista sale vacía y no hay error

El selector no corresponde a nada. Es el más traicionero. Abre con la ventana
visible, haz clic derecho sobre el dato → *Inspeccionar*, y compara.

## Resumen

| Para | Instrucción |
|---|---|
| Un elemento | `find_element(By.X, "...")` — falla si no está |
| Varios | `find_elements(By.X, "...")` — lista vacía si no está |
| Buscar dentro de otro | `elemento.find_elements(...)` |
| Lo que se ve | `.text` |
| Lo que no se ve | `.get_attribute("href")` |
| Qué etiqueta es | `.tag_name` |
| Cerrar | `.quit()` |

### Las tres ideas que hay que llevarse

1. **Un WebElement no es texto.** Es una referencia a algo en la página; el
   texto se lo pides con `.text`.
2. **No generalices desde un ejemplo.** Compara siempre cinco o seis casos
   antes de dar por buena una regla. El `h3[2]` parecía la ubicación.
3. **Busca por contenido cuando la posición pueda moverse.** Un índice que
   falla no avisa: guarda el dato equivocado en silencio.

---
## Para practicar

1. Agrega una columna con la **calificación de la empresa** (los `3.1`, `4.4`).
   Pista: se reconoce con `re.fullmatch(r"\d+\.\d+", texto)`.
2. Separa `ubicacion` en `distrito` y `ciudad`.
3. ¿Cuántas ofertas dicen "Apto discapacidad"? Búscalo en los `<h3>`.